# Advanced Python Concepts

## Decorators

A decorator is a function that manipulates a function or a class. It's a very handy element of the Python language.

 We'll see in this lab how to use them.

### Function Decorators: Functions That Manipulate Functions

Write a function `squared` that takes a function as input and returns that same function with its result squared:

- the original function always takes two arguments
- the original function always produces a number
- the squared function takes **a function** as input and produces one as output

In [ ]:
def add(a, b):
  return a + b


def sub(a, b):
  return a - b


def mul(a, b):
  return a * b


def squared(function):
  return function  # Your code here


print(squared(add)(2, 2))  # Should be 16
print(squared(sub)(10, 5))  # Should be 25
print(squared(mul)(3, 4))  # Should be 144

#### Solution

In [ ]:
def squared(function):
  def new_function(a, b):
    return function(a, b) ** 2

  return new_function


print(squared(add)(2, 2))
print(squared(sub)(10, 5))
print(squared(mul)(3, 4))

### Decorator Syntax

To apply a decorator to a function, you can use the following syntax:

In [ ]:
@squared
def div(a, b):
  return a / b


print(div(8, 4))

Without this `@` syntax, how would you get the same result?

In [ ]:
# Your code here

#### Solution

In [ ]:
def div(a, b):
  return a / b


div = squared(div)


print(div(8, 4))

### Measuring a Function's Execution Time

Write a `timeit` decorator that displays a function's execution time. You can use the [`time.time`](https://docs.python.org/3/library/time.html#time.time) function

In [ ]:
# Your code here

#### Solution

In [ ]:
import time


def timeit(f):
  def new_f(*args, **kwargs):
    start = time.time()
    result = f(*args, **kwargs)
    print(f"Time elapsed during '{f.__name__}' call: {time.time() - start}s")
    return result

  return new_f


@timeit
def add(a, b):
  return a + b


add(1, 2)

### Decorator With Arguments

Sometimes, it's useful to be able to give arguments to a decorator.

To do so, you need to create an extra "layer" of function: we'll define a function with arguments that returns a decorator without arguments (so itself a function that takes a function as argument… it's not easy).

See the following example, which adds to the `timeit` decorator seen previously an argument to display the result in seconds or milliseconds:

In [ ]:
import time


def timeit(unit="s"):
  def decorator(function):
    def new_f(*args, **kwargs):
      start = time.time()
      result = function(*args, **kwargs)
      duration = time.time() - start
      if unit == "ms":
        duration *= 1000
      elif unit != "s":
        raise ValueError("Can only use s or ms as unit argument")
      print(f"Time elapsed during '{function.__name__}' call: {duration}{unit}")
      return result

    return new_f

  return decorator


@timeit(unit="ms")
def add(a, b):
  return a + b


add(1, 2)

### Automatic Logging

Following a similar pattern, automatically log calls to a function, with the arguments used. You can use the [`print`](https://docs.python.org/3/library/functions.html#print) function or the functions of the [`logging`](https://docs.python.org/3/library/logging.html) module

In [ ]:
# Your code here

#### Solution

In [ ]:
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def autolog(logger):
  def decorator(f):
    def new_f(*args, **kwargs):
      result = f(*args, **kwargs)
      logger.info("%s called with pos args %s and kwargs %s", f.__name__, args, kwargs)
      return result

    return new_f

  return decorator


@autolog(logger)
def add(a, b):
  return a + b


add(1, b=2)

### Class Registry

It's often useful to register classes or functions in a data structure (a registry), for instance to offer a plugin system.

Write a class decorator that registers each class it's applied to in a dictionary, under the name given as its argument.

In [ ]:
# Your code here

Besides adding the decorator to the classes you want to register, what important step is needed for the registration to happen?

*Your answer*

#### Solution

In [ ]:
from collections.abc import Callable
from typing import Any

_plugins = {}


def register(name: str) -> Callable[[type[Any]], type[Any]]:
  def decorator(cls: type[Any]) -> type[Any]:
    _plugins[name] = cls
    return cls

  return decorator


@register("A")
class A:
  pass

The module containing the class must be loaded. This can be done automatically to make plugin systems easier to use, as in the AllenNLP library, for instance, which has a function to load all the modules of a given package: [`import_module_and_submodules`](https://github.com/allenai/allennlp/blob/a0edfae9ca571ed7d43749974bb842167201c2da/allennlp/common/util.py#L331)

### Using [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps)

Try manipulating the functions modified in the rest of the lab and finding their name (`__name__` attribute) or their doc (`__doc__` attribute). What do you notice?

*Your answer*

Use `functools.wraps` on the given example of the `timeit` decorator with an argument to fix this problem.

In [ ]:
# Your code here

#### Solution

In [ ]:
import functools
import time


def timeit(unit="s"):
  def decorator(function):
    @functools.wraps(function)
    def new_f(*args, **kwargs):
      start = time.time()
      result = function(*args, **kwargs)
      duration = time.time() - start
      if unit == "ms":
        duration *= 1000
      elif unit != "s":
        raise ValueError("Can only use s or ms as unit argument")
      print(f"Time elapsed during '{function.__name__}' call: {duration}{unit}")
      return result

    return new_f

  return decorator


@timeit(unit="ms")
def add(a, b):
  """Add two integers."""
  return a + b


print(add.__name__, add.__doc__)

## Closures

### Defining a Closure

Implement a closure that, when called, returns the time elapsed since the closure was created.

In [ ]:
# Your code here

#### Solution

In [ ]:
from collections.abc import Callable
from time import sleep, time


def make_timer() -> Callable[[], float]:
  start = time()

  def timer() -> float:
    return time() - start

  return timer


timer = make_timer()
sleep(2)
print(timer())
sleep(1)
print(timer())

## Generators

### Sequence Transformers

Generators are widely used to transform and use sequences without ever storing them entirely.

Write the following transformers, which will process sequences of integers:

- `double`, which doubles the values given to it
- `filter_even`, which removes the odd numbers from the sequence given to it

Test your transformers on a large sequence. Watch the instance's memory usage.

In [ ]:
# Your code here

#### Solution

In [ ]:
from collections.abc import Iterable, Iterator


def double(iterator: Iterable[int]) -> Iterator[int]:
  for item in iterator:
    yield item * 2


def filter_even(iterator: Iterable[int]) -> Iterator[int]:
  for item in iterator:
    if item % 2:
      yield item


for i in filter_even(double(range(100_000_000))):
  pass

### Infinite Date Generator

Write the `gen_dates` function which, from a given date, generates all the successive dates one week apart.

In [ ]:
import datetime
from collections.abc import Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  pass  # Your code here

Suggest a direct way to use this infinite generator.

Also suggest a method `take(iterator: Iterator[T], n: int) -> Iterator[T]` (which takes an iterator as input and modifies it) to use this infinite generator.

In [ ]:
# Your code here

#### Solution

In [ ]:
import datetime
from collections.abc import Iterable, Iterator
from typing import TypeVar


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    date += week
    yield date


for i, d in enumerate(gen_dates(datetime.date(1900, 1, 1))):
  if i == 10:
    break
  print(d)


T = TypeVar("T")


def take(iterator: Iterable[T], n: int) -> Iterator[T]:
  for i, item in enumerate(iterator):
    if i == n:
      break
    yield item


for date in take(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

### Using a Generator's `send` Method (Very Hard)

Design a generator that, given a starting amount of money, receives an interest rate from you and gives you back the new value of the amount after applying the rate.

In [ ]:
# Your code here

#### Solution

In [ ]:
from collections.abc import Generator


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    rate = yield amount
    amount *= rate


amounts = interests(100)
amounts.send(None)  # Initialize the generator
for rate in [1.05, 1.08, 0.98]:
  print(amounts.send(rate))

### Using a Generator's `throw` Method (Very Hard)

Start from the code of the previous exercise and add a mechanism to add an amount of money (or remove some with a negative amount). To do so, you can use `throw`, the type of the exception acting as a message for the generator.

In [ ]:
# Your code here

#### Solution

In [ ]:
import pathlib
from collections.abc import Generator


class AddAmountException(Exception):
  def __init__(self, value):
    super().__init__(value)
    self.value = value


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    try:
      rate = yield amount
      amount *= rate
    except AddAmountException as e:
      amount += e.value


amounts = interests(100)
amounts.send(None)  # Initialize the generator
print(amounts.send(1.05))
print(amounts.send(1.08))
amounts.throw(AddAmountException(100))
print(amounts.send(0.98))

## Comprehensions

For this section, reimplement the various loops given using comprehensions.

### Simple Comprehension

In [ ]:
d = {"a": [1, 2, 3], "b": [1, 2, 3, 4, 5]}

lengths = {}
for k, v in d.items():
  lengths[k] = len(v)
print(lengths)

In [ ]:
# Your code here

#### Solution

In [ ]:
lengths = {k: len(v) for k, v in d.items()}
print(lengths)

### Comprehensions With Multiple Iterations

In [ ]:
a_list = [1, 2, 3]
b_list = ["a", "b", "c"]
cartesian_product = []
for a in a_list:
  for b in b_list:
    cartesian_product.append((a, b))
print(cartesian_product)

In [ ]:
# Your code here

#### Solution

In [ ]:
cartesian_product = [(a, b) for a in a_list for b in b_list]
print(cartesian_product)

### File

In [ ]:
with pathlib.Path("sample_data/README.md").open(encoding="utf8") as fh:
  total = 0
  for line in fh:
    total += len(line)
print(total)

In [ ]:
# Your code here

#### Solution

In [ ]:
with pathlib.Path("sample_data/README.md").open(encoding="utf8") as fh:
  total = sum(len(line) for line in fh)
print(total)

# Your code here